[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C23_Frontier_Alignment_Course/03_scalable_oversight/03_scalable_oversight.ipynb)

# 03 · 可扩展监督与辩论（用 numpy 模拟）

目标：用纯 numpy 模拟 **debate**（弱裁判 + 强辩手, 准确率随论证质量上升）、**混淆论证**如何破坏它、**sandwiching** 怎么度量协议增益、以及 **IDA** 的分解-放大。

路线：辩论博弈 → 准确率曲线 → 混淆论证 → sandwiching 增益 → RRM bootstrap → IDA 分解 → ✏️ 练习 → 📖 答案 → 🧪 实证胶囊。

> 心智模型：**真理方论证更强（结构优势）→ 弱裁判按证据强度差判胜 → 借对抗放大监督。** 我们知道每题真答案, 所以能精确量化「裁判借辩论判对的概率」。

## 1 · 一场玩具辩论

一个问题有真答案。**诚实辩手**论证指向真答案、论证质量 `q_h`；**撒谎辩手**指向假答案、质量 `q_l`。
每轮各给一条带强度的可验证证据（诚实方强度均值更高 = 真理的结构优势）。**弱裁判**按双方证据强度差的 sigmoid 概率判胜。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
def sigmoid(z): return 1.0/(1.0+np.exp(-np.clip(z, -30, 30)))

def run_one_debate(q_honest, q_liar, rounds, rng):
    '''诚实方/撒谎方每轮证据 ~ N(质量,1), 累加。裁判按强度差 sigmoid 概率判「诚实方胜」。'''
    h = rng.normal(q_honest, 1.0, size=rounds).sum()   # 诚实方总证据强度
    l = rng.normal(q_liar,   1.0, size=rounds).sum()   # 撒谎方总证据强度
    p_judge_picks_honest = sigmoid(h - l)
    return rng.random() < p_judge_picks_honest          # True = 裁判判对(选了诚实方/真答案)

# 单场演示
wins = [run_one_debate(1.5, 0.0, rounds=3, rng=rng) for _ in range(10)]
print('10 场辩论, 裁判判对(选真答案):', sum(wins), '/ 10')
assert isinstance(wins[0], (bool, np.bool_))
print('✅ 辩论协议跑通：弱裁判借「证据强度差」判优劣')

## 2 · 核心曲线：裁判准确率随诚实方论证质量上升

把撒谎方质量固定为 0, 扫诚实方质量 `q_h`, 统计裁判判对的比例。
**这是 debate 的立身之本**：诚实方论证越强（真理的结构优势越大）, 弱裁判越能选出真答案 —— 即使裁判本身不会解题。

In [ ]:
def judge_accuracy(q_honest, q_liar, rounds, rng, trials=3000):
    correct = sum(run_one_debate(q_honest, q_liar, rounds, rng) for _ in range(trials))
    return correct / trials

qs = [0.0, 0.5, 1.0, 1.5, 2.0, 3.0]
accs = [judge_accuracy(q, 0.0, rounds=3, rng=rng) for q in qs]
print(f"{'诚实方质量 q_h':>14}{'裁判准确率':>12}")
for q, a in zip(qs, accs):
    print(f'{q:>14.1f}{a:>12.3f}')
# q_h=0(无优势)时裁判≈瞎猜0.5; q_h 增大, 准确率单调升向 1
assert abs(accs[0] - 0.5) < 0.05, '无论证优势时裁判≈随机'
assert all(accs[i+1] >= accs[i] - 0.03 for i in range(len(accs)-1)), '准确率应随 q_h 上升'
assert accs[-1] > 0.95, '强论证优势下弱裁判也能高准确率'
print('\n✅ 关键曲线：裁判准确率随诚实方论证质量单调上升(0.5 -> ~1.0)')

**这正是 Khan et al. 2024 在真实 LLM 辩论上的观测**：更强（更有说服力）的辩手, 让较弱的裁判更常判对。
弱裁判的能力没变, 但通过对抗结构, 它「借」到了强辩手的能力。

## 3 · debate 的命门：混淆论证

debate 押注「揭穿比欺骗容易」。但若撒谎方能制造**人类难以证伪的论证**, 把自己的「证据强度」伪装得和诚实方一样高, 诚实方的优势就被**抹平**, 裁判退回随机。我们模拟：让 `q_liar` 逼近 `q_honest`, 看准确率塌回 0.5。

In [ ]:
q_h = 2.0
print(f"{'撒谎方质量 q_l':>14}{'裁判准确率':>12}{'说明':>16}")
for q_l in [0.0, 0.5, 1.0, 1.5, 2.0]:
    a = judge_accuracy(q_h, q_l, rounds=3, rng=rng)
    note = '诚实方占优' if q_l < q_h - 0.3 else '混淆!被抹平'
    print(f'{q_l:>14.1f}{a:>12.3f}{note:>16}')
acc_clean = judge_accuracy(q_h, 0.0, rounds=3, rng=rng)
acc_obf   = judge_accuracy(q_h, q_h,  rounds=3, rng=rng)   # 撒谎方完全混淆
print(f'\n诚实方占优时准确率 {acc_clean:.3f}; 完全混淆时 {acc_obf:.3f}')
assert acc_clean > 0.9, '诚实方占优时裁判可靠'
assert abs(acc_obf - 0.5) < 0.06, '完全混淆把准确率抹回随机'
print('✅ 混淆论证现形：撒谎方匹配证据强度 -> 真理优势消失 -> 裁判退回瞎猜')

**这是 debate 尚未解决的开放难题**（Barnes & Christiano 2020）：存在一类问题, 谎言可伪装成「同样难验证」, 抹平「验证比生成易」的不对称。超人模型恰可能擅长制造这类论证 —— 所以 debate 至今是**研究问题而非成熟方案**。

## 4 · sandwiching：度量协议带来的监督增益

怎么知道辩论协议**真的**有用？用 sandwiching：比较三层 —— **底层**(裁判独自瞎猜≈0.5)、**中层**(裁判 + 辩论协议)、**顶层**(专家=真答案, 准确率1.0)。
**gap recovered** = (协议 − 底层) / (顶层 − 底层)：协议把弱裁判推到多接近专家。

In [ ]:
baseline = 0.5      # 弱裁判独自(二选一瞎猜)
expert   = 1.0      # 专家(=真答案)

def gap_recovered(protocol_acc, baseline=0.5, expert=1.0):
    return (protocol_acc - baseline) / (expert - baseline)

# 协议 = 诚实方有中等论证优势的辩论
protocol_acc = judge_accuracy(1.5, 0.0, rounds=3, rng=rng)
gr = gap_recovered(protocol_acc)
print(f'底层(瞎猜)   = {baseline:.2f}')
print(f'中层(辩论)   = {protocol_acc:.3f}')
print(f'顶层(专家)   = {expert:.2f}')
print(f'gap recovered = {gr:.3f}  (辩论把弱裁判推到专家与瞎猜之间的 {gr:.0%})')
assert 0.0 < gr <= 1.05, 'gap recovered 应在 (0,1]'
assert protocol_acc > baseline, '协议应优于瞎猜基线'
print('✅ sandwiching 把「协议有没有用」变成可测量的 gap recovered')

## 5 · recursive reward modeling：监督能力逐级 bootstrap

RRM：用上一级造出的 AI 助手, 去监督这一级更难的任务。我们用玩具模拟：每一级的「可监督难度」随上一级助手能力增长, 看监督前沿如何一级级推进。

In [ ]:
def rrm_bootstrap(levels, base_capability=1.0, gain_per_level=0.8):
    '''每级: 助手能力 = 上级能力 + 增益; 可监督难度 = 当前助手能力。返回每级可监督难度。'''
    capability = base_capability
    frontier = []
    for lv in range(levels):
        frontier.append(capability)        # 这一级能监督到的难度 = 当前能力
        capability += gain_per_level        # 用这一级造出的助手, 下一级能力更强
    return frontier

frontier = rrm_bootstrap(5)
print('各级可监督难度前沿:', [round(f,1) for f in frontier])
assert all(frontier[i+1] > frontier[i] for i in range(len(frontier)-1)), '监督前沿应逐级推进'
assert frontier[-1] > frontier[0], '最终能监督的难度远超起点'
print('✅ RRM: 监督能力像搭脚手架一样逐级抬升(每级建立在上级造出的助手上)')

## 6 · iterated amplification：分解-放大

IDA 的核心动作：把难题**分解**成弱解题器能处理的子问题, 分别解, 再**聚合**。
玩具版：弱解题器只会求和 ≤3 个数(能力有限), 但通过递归分解-聚合, 能正确求和任意多个数 —— 能力被放大。

In [ ]:
def weak_solver(xs):
    '''弱解题器: 只能正确处理 <=3 个数。'''
    assert len(xs) <= 3, '弱解题器一次只能处理 3 个'
    return sum(xs)

def amplified_solver(xs, chunk=3):
    '''分解-放大: 切成 <=3 的块, 各自弱解, 再递归聚合部分和。'''
    if len(xs) <= chunk:
        return weak_solver(xs)
    partials = [weak_solver(xs[i:i+chunk]) for i in range(0, len(xs), chunk)]
    return amplified_solver(partials, chunk)

arr = list(range(1, 13))                # 12 个数, 远超弱解题器的 3 个上限
result = amplified_solver(arr)
print(f'放大求解 sum(1..12) = {result}, 真值 = {sum(arr)}')
assert result == sum(arr), '分解-放大应得到正确总和'
# 弱解题器单独无法处理 12 个
try:
    weak_solver(arr); raise RuntimeError('不应到这')
except AssertionError:
    print('弱解题器单独处理 12 个 -> 拒绝(超能力)')
print('✅ IDA: 分解-聚合把弱能力放大到解决远超单步能力的问题')

---
## ✏️ 练习 1：多轮辩论的胜负判定 `debate_winner`

实现 `debate_winner(honest_evidence, liar_evidence)`：给定两方各轮证据强度的**列表**, 按总强度判胜, 返回 `'honest'` 或 `'liar'`（总强度相等判 `'honest'`, 真理优先）。

In [ ]:
def debate_winner(honest_evidence, liar_evidence):
    # TODO: 比较 sum(honest_evidence) 与 sum(liar_evidence), 返回胜方字符串
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert debate_winner([1.0, 2.0, 1.5], [0.5, 0.5, 0.5]) == 'honest'
assert debate_winner([0.1, 0.1], [2.0, 2.0]) == 'liar'
assert debate_winner([1.0, 1.0], [1.0, 1.0]) == 'honest', '平手真理优先'
print('✅ 练习 1 通过：能按总证据强度判辩论胜负')

## ✏️ 练习 2：准确率随轮数的变化 `accuracy_vs_rounds`

更多轮 = 更多独立证据 = 诚实方优势累积得更明显。
实现 `accuracy_vs_rounds(q_h, q_l, round_list, rng, trials)`：对每个轮数算裁判准确率, 返回准确率列表。验证准确率随轮数上升（更多轮放大真理优势）。

In [ ]:
def accuracy_vs_rounds(q_h, q_l, round_list, rng, trials=3000):
    # TODO: 对 round_list 里每个 rounds, 用 judge_accuracy 算准确率, 返回列表
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
accs_r = accuracy_vs_rounds(0.6, 0.0, [1, 3, 6, 12], np.random.default_rng(3))
print('轮数 [1,3,6,12] 的准确率:', [round(a,3) for a in accs_r])
assert accs_r[-1] > accs_r[0], '更多轮应放大诚实方优势, 准确率上升'
assert accs_r[0] > 0.5, '哪怕1轮也优于随机(诚实方有优势)'
print('✅ 练习 2 通过：更多轮辩论放大真理优势')

## ✏️ 练习 3：sandwiching 的协议比较 `compare_protocols`

你有两个协议各自的中层准确率。实现 `compare_protocols(accs, baseline, expert)`：
返回一个 dict, 把每个协议名映射到它的 gap recovered, 用于排序哪个协议更可扩展。

In [ ]:
def compare_protocols(accs, baseline=0.5, expert=1.0):
    # TODO: accs 是 {协议名: 准确率}; 返回 {协议名: gap_recovered}
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
accs = {'no_debate': 0.55, 'debate': 0.90, 'debate+critique': 0.95}
grs = compare_protocols(accs)
assert abs(grs['debate'] - 0.8) < 1e-9, '(0.9-0.5)/(1-0.5)=0.8'
assert grs['debate+critique'] > grs['debate'] > grs['no_debate']
best = max(grs, key=grs.get)
print('各协议 gap recovered:', {k: round(v,2) for k,v in grs.items()})
print(f'最可扩展的协议: {best}')
print('✅ 练习 3 通过：能用 gap recovered 排序协议')

## ✏️ 练习 4：IDA 分解求最大值 `amplified_max`

把第 6 节的分解-放大从「求和」改成「求最大值」。弱解题器只会比较 ≤3 个数的最大值。
实现 `amplified_max(xs, chunk=3)`：递归分解-聚合, 正确求出任意多个数的最大值。

In [ ]:
def amplified_max(xs, chunk=3):
    # TODO: 仿照 amplified_solver, 但用 max 而非 sum; 弱步只处理 <=3 个
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
import random as _r
data = [_r.Random(7).randint(0, 100) for _ in range(20)]
data = [3, 41, 12, 99, 7, 55, 88, 2, 64, 30, 1, 77, 45, 22, 90, 13, 6, 81, 50, 38]
assert amplified_max(data) == max(data), '分解求最大值应正确'
assert amplified_max([5]) == 5 and amplified_max([2, 9, 4]) == 9
print(f'放大求 max = {amplified_max(data)}, 真值 = {max(data)}')
print('✅ 练习 4 通过：分解-放大同样适用于求最大值(任何可结合的聚合)')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def debate_winner(honest_evidence, liar_evidence):
    return 'honest' if sum(honest_evidence) >= sum(liar_evidence) else 'liar'

In [ ]:
# 练习 2 参考答案
def accuracy_vs_rounds(q_h, q_l, round_list, rng, trials=3000):
    return [judge_accuracy(q_h, q_l, r, rng, trials) for r in round_list]

In [ ]:
# 练习 3 参考答案
def compare_protocols(accs, baseline=0.5, expert=1.0):
    return {name: (a - baseline) / (expert - baseline) for name, a in accs.items()}

In [ ]:
# 练习 4 参考答案
def amplified_max(xs, chunk=3):
    if len(xs) <= chunk:
        assert len(xs) <= chunk
        return max(xs)
    partials = [max(xs[i:i+chunk]) for i in range(0, len(xs), chunk)]
    return amplified_max(partials, chunk)

---
## 🧪 真实数据胶囊：Khan et al. 2024 的辩论实证

Khan et al. 2024《Debating with More Persuasive LLMs Leads to More Truthful Answers》在真实 LLM 辩论上发现：**更有说服力(更强)的辩手, 让裁判的准确率更高**。我们用论文报告**量级**的数字, 验证这个单调关系。

（真实数字随模型/任务变化, 这里用代表性量级演示趋势。）

In [ ]:
# 论文趋势(量级): 辩手说服力(persuasiveness) 越高, 裁判准确率越高
# (代表性数字, 体现单调上升的关系)
PERSUASIVENESS = [0.0, 0.3, 0.6, 0.9]    # 辩手相对说服力(归一化)
JUDGE_ACC      = [0.52, 0.64, 0.72, 0.76] # 对应裁判准确率(量级)
print(f"{'辩手说服力':>10}{'裁判准确率':>12}")
for pp, aa in zip(PERSUASIVENESS, JUDGE_ACC):
    print(f'{pp:>10.1f}{aa:>12.2f}')
# 单调上升 + 都优于随机 = 支持「debate 帮助弱裁判」
assert all(JUDGE_ACC[i+1] > JUDGE_ACC[i] for i in range(len(JUDGE_ACC)-1)), '应单调上升'
assert JUDGE_ACC[0] >= 0.5, '至少不差于随机'
print('\n观察: 与我们的玩具曲线一致 —— 更强辩手 -> 裁判更常判对')

**🧪 胶囊练习**：实现 `persuasiveness_acc_corr(persuasiveness, judge_acc)`：用 `np.corrcoef` 算两者相关系数, 验证 debate 的核心假设（辩手越强、裁判越准）在数据上成立（相关系数 > 0.9）。

In [ ]:
def persuasiveness_acc_corr(persuasiveness, judge_acc):
    # TODO: 返回两个序列的 Pearson 相关系数(np.corrcoef[0,1])
    raise NotImplementedError

In [ ]:
# 自测
r = persuasiveness_acc_corr(PERSUASIVENESS, JUDGE_ACC)
assert r > 0.9, '说服力与准确率应强正相关'
print(f'说服力 vs 裁判准确率 相关系数 = {r:.3f}')
print('✅ 胶囊练习通过：实证支持 debate 核心假设(更强辩手 -> 更准裁判)')

In [ ]:
# 📖 胶囊参考答案
def persuasiveness_acc_corr(persuasiveness, judge_acc):
    return float(np.corrcoef(persuasiveness, judge_acc)[0, 1])

### 小结
- **可扩展监督**：监督者 < 被监督者时, 靠**协议**(而非更强监督者)从弱监督榨出可信信号。
- **debate**：真理方论证有结构优势 -> 弱裁判按证据差判胜 -> 准确率随论证质量/轮数上升。押注「验证比生成易」。
- **混淆论证**：谎言伪装成「同样难验证」-> 抹平优势 -> 裁判退回随机。debate 未解决的命门。
- **RRM/IDA**：用上级造出的助手监督更难任务(bootstrap) / 分解-聚合放大弱能力。
- **sandwiching**：用「专家 vs 非专家」预演「超人 vs 人类」, gap recovered 度量协议增益。

下一站：**模块 04 · Weak-to-Strong** —— 一个更直接的乐观信号: 弱监督能否「引出」强模型本就具备的能力。